# Jet and wax-oil (VGO) hydrotreating

`difflow_refinery.hydrotreating.services` sets the existing hydrotreater up for two services:

- **jet/kerosene:** a mild operating point, with the product checked against the jet fuel specs;
- **wax oil (VGO):** VGO kinetics, then Conradson carbon and metals removal, with the product handed to the FCC as its feed.

Each hydrotreater solve compiles for about a minute. The rate constants are illustrative throughout, as in the hydrotreater itself.

In [1]:
import warnings
import jax
import jax.numpy as jnp
import difflow_refinery as dr
from difflow_refinery.assay import default_cut_points
from difflow_refinery.hydrotreating import Hydrotreater
from difflow_refinery.hydrotreating.feed import select_cuts, straight_run_cut
from difflow_refinery.hydrotreating.services import (fcc_feed_from_hydrotreater, heavy_ends_removal,
                                                     jet_margins, jet_params, jet_properties, vgo_params)
from difflow_refinery.fcc import FCCParams, FCCUnit

jax.config.update("jax_enable_x64", True)
warnings.simplefilter("ignore")

assay = dr.Assay([5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95],
                 jnp.asarray([t + 273.15 for t in (60, 95, 150, 205, 260, 315, 370, 430, 500, 600, 680)]),
                 sg=0.86, light_ends={"propane": 0.5, "n_butane": 1.0, "n_pentane": 1.5},
                 heavy_end=dr.HeavyEnd(), sulfur_wt=1.8, nitrogen_wppm=1500.0, ccr_wt=2.0,
                 nickel_vanadium_wppm=40.0)
char = dr.characterize(assay, cut_points=tuple(float(c) for c in default_cut_points(assay)),
                       composition=True)

def cut(lo_C, hi_C):
    return straight_run_cut(char, rate=50.0, cuts=select_cuts(char, lo_C + 273.15, hi_C + 273.15))

## Jet

A 150-250 C kerosene cut at the mild jet point. The margins are signed, so a negative one is off spec. The freeze point needs per-component n-paraffins, which the outputs do not carry, so it is NaN and never counts as met. Mercaptan sulfur is reported as the total sulfur, an upper bound.

In [2]:
kero = cut(150, 250)
res = Hydrotreater(char, kero, jet_params()).solve(kero)
print("converged", bool(res.converged), " balances", {k: f"{float(v):.0e}" for k, v in res.balances.items()})
jp = jet_properties(res.outputs)
for k, m in jet_margins(jp).items():
    print(f"{k:18s} {float(jp[k]):9.2f}   margin {float(m):9.2f}")

converged True  balances {'C': '3e-16', 'H': '1e-16', 'N': '2e-16', 'S': '2e-16', 'mass': '1e-16'}
smoke_point_mm         23.33   margin     -1.67
flash_point_C          48.47   margin     10.47
freeze_point_C           nan   margin       nan
aromatics_vol          26.67   margin     -1.67
S_wppm               1383.21   margin   1616.79
mercaptan_S_wppm     1383.21   margin  -1353.21


## Wax oil to the FCC

A 370-560 C wax oil through the VGO point. Conradson carbon and metals come off after the reactor, through an illustrative first-order correlation. The treated product then becomes the FCC's feed.

In [3]:
vgo = cut(370, 560)
res = Hydrotreater(char, vgo, vgo_params()).solve(vgo)
o = res.outputs
print("HDS", float(o["hds.conversion"]), " HDN", float(o["hdn.conversion"]), " WABT", float(o["wabt"]) - 273.15, "C")
names = select_cuts(char, 370 + 273.15, 560 + 273.15)
idx = jnp.asarray([char.names.index(n) for n in names])
mass = jnp.asarray([vgo[f"F_{n}"] for n in names]) * jnp.asarray(char.component_MW)[idx]
ccr_in = float(100 * (mass * jnp.asarray(char.ccr)[idx]).sum() / mass.sum())
niv_in = float(1e6 * (mass * jnp.asarray(char.nickel_vanadium)[idx]).sum() / mass.sum())
he = heavy_ends_removal(ccr_in, niv_in, o["wabt"], vgo_params().lhsv)
print(f"CCR {ccr_in:.3f} -> {float(he['ccr_wt']):.3f} wt%,  Ni+V {niv_in:.3f} -> {float(he['niv_wppm']):.4f} wppm")
feed = fcc_feed_from_hydrotreater(o, ccr_wt=he["ccr_wt"], niv_wppm=he["niv_wppm"])
fcc = FCCUnit(FCCParams()).solve(feed)
print("FCC converged", bool(fcc["converged"]),
      {k: round(float(fcc["outputs"][k]), 4) for k in ("conversion", "yield.gasoline", "yield.coke")})

HDS 0.9999958594693855  HDN 0.9999999997175498  WABT 411.91211679147364 C
CCR 0.656 -> 0.328 wt%,  Ni+V 0.156 -> 0.0092 wppm


FCC converged True {'conversion': 0.7759, 'yield.gasoline': 0.4937, 'yield.coke': 0.047}
